# Evaluation Notebook

Runs the full local eval suite on all model variants.

**Before running:**
1. Enable GPU: Notebook settings → Accelerator → **T4 x2** or **P100**
2. Enable Internet: Notebook settings → Internet → **On**
3. Add HuggingFace token: Add-ons → Secrets → **HF_TOKEN** (toggle On)
4. Add notebook outputs as input data:
   - **dpo-with-refusal** (your off-policy DPO notebook)
   - **on-policy-dpo-refusal** (your on-policy DPO notebook)

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes datasets

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
print("HF login OK")

In [ ]:
REPO_URL = "https://github.com/202520030411/Distill-Safety-Aligned-Models.git"
REPO_DIR = "/kaggle/working/repo"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

print("Repo cloned.")
!ls {REPO_DIR}/outputs/

## Copy New Adapters from Kaggle Inputs

Finds the `dpo_with_refusals` and `on_policy_dpo_with_refusals` adapters from
the other notebook outputs you added as input data.

In [ ]:
import glob, shutil

def find_and_copy_adapter(name, search_patterns):
    """Search for an adapter_config.json in Kaggle input dirs and copy the whole adapter."""
    dest = f"{REPO_DIR}/outputs/{name}"
    if os.path.exists(f"{dest}/adapter_config.json"):
        print(f"  {name}: already exists at {dest}")
        return True
    for pattern in search_patterns:
        matches = glob.glob(pattern, recursive=True)
        if matches:
            src_dir = os.path.dirname(matches[0])
            shutil.copytree(src_dir, dest, dirs_exist_ok=True)
            print(f"  {name}: copied from {src_dir}")
            return True
    print(f"  {name}: NOT FOUND in /kaggle/input/")
    return False

print("Looking for new adapters...")

find_and_copy_adapter("dpo_with_refusals", [
    "/kaggle/input/**/dpo_with_refusals/adapter_config.json",
    "/kaggle/input/**/outputs/dpo_with_refusals/adapter_config.json",
])

find_and_copy_adapter("on_policy_dpo_with_refusals", [
    "/kaggle/input/**/on_policy_dpo_with_refusals/adapter_config.json",
    "/kaggle/input/**/outputs/on_policy_dpo_with_refusals/adapter_config.json",
])

print("\nAll adapters:")
!ls {REPO_DIR}/outputs/

## Patch eval/common.py with New Model Specs

The GitHub repo may not have the updated eval code yet, so we add the new model specs here.

In [ ]:
common_path = f"{REPO_DIR}/eval/common.py"
with open(common_path, "r") as f:
    content = f.read()

if "dpo_with_refusals" not in content:
    patch = '''    "dpo_with_refusals": ModelSpec(
        name="dpo_with_refusals",
        model_id="meta-llama/Llama-3.2-1B",
        adapter_path=PROJECT_ROOT / "outputs" / "dpo_with_refusals",
        merged_adapter_path=PROJECT_ROOT / "outputs" / "with_refusals",
    ),
    "on_policy_dpo_with_refusals": ModelSpec(
        name="on_policy_dpo_with_refusals",
        model_id="meta-llama/Llama-3.2-1B",
        adapter_path=PROJECT_ROOT / "outputs" / "on_policy_dpo_with_refusals",
        merged_adapter_path=PROJECT_ROOT / "outputs" / "with_refusals",
    ),
}'''
    old_end = '''    "on_policy_dpo": ModelSpec(
        name="on_policy_dpo",
        model_id="meta-llama/Llama-3.2-1B",
        adapter_path=PROJECT_ROOT / "outputs" / "on_policy_dpo",
        merged_adapter_path=PROJECT_ROOT / "outputs" / "baseline",
    ),
}'''
    new_end = '''    "on_policy_dpo": ModelSpec(
        name="on_policy_dpo",
        model_id="meta-llama/Llama-3.2-1B",
        adapter_path=PROJECT_ROOT / "outputs" / "on_policy_dpo",
        merged_adapter_path=PROJECT_ROOT / "outputs" / "baseline",
    ),
''' + patch
    content = content.replace(old_end, new_end)
    with open(common_path, "w") as f:
        f.write(content)
    print("Patched eval/common.py with dpo_with_refusals + on_policy_dpo_with_refusals")
else:
    print("eval/common.py already has new model specs")

## Run Evaluation

Evaluates each model on 4 metrics: safety, refusal, quality, jailbreak.

~5–8 min per 1B model, ~15–20 min for teacher (8B).

In [ ]:
MODELS_TO_EVAL = [
    "with_refusals",
    "dpo_with_refusals",
    "on_policy_dpo_with_refusals",
]

model_str = " ".join(MODELS_TO_EVAL)
print(f"Evaluating: {model_str}")
print(f"Results will be saved to {REPO_DIR}/results/eval/")

In [ ]:
os.chdir(f"{REPO_DIR}/eval")
!python run_all.py --models {model_str}

## View Results

In [ ]:
import json

summary_path = f"{REPO_DIR}/results/eval/summary.json"
with open(summary_path) as f:
    summary = json.load(f)

print(f"{'Model':<35} {'Unsafe':>8} {'FalseRef':>8} {'Quality':>8} {'Jailbrk':>8}")
print("-" * 75)
for model, metrics in summary.items():
    print(f"{model:<35} "
          f"{metrics['unsafe_compliance_rate']:>8.3f} "
          f"{metrics['false_refusal_rate']:>8.3f} "
          f"{metrics['reference_quality_proxy']:>8.3f} "
          f"{metrics['jailbreak_unsafe_compliance_rate_macro']:>8.3f}")

## Download Results

In [ ]:
shutil.make_archive("/kaggle/working/eval_results", "zip",
                    f"{REPO_DIR}/results", "eval")
print("Download eval_results.zip from the Output tab.")